In [1]:
import pyautogui as pyg
import time

print("【座標測量倒數開始】")
print("請在 4 秒內,將滑鼠游標移到遊戲左下角『HINT』按鈕的正中央,並保持不動！")
print("4...")
time.sleep(1)
print("3...")
time.sleep(1)
print("2...")
time.sleep(1)
print("1...")
time.sleep(1)

# 取得目前滑鼠的精確位置
real_x, real_y = pyg.position()

print("=" * 40)
print(f"🎯 測量成功！你電腦上的 HINT 按鈕座標為：")
print(f"hint_x = {real_x}")
print(f"hint_y = {real_y}")
print("=" * 40)

【座標測量倒數開始】
請在 4 秒內,將滑鼠游標移到遊戲左下角『HINT』按鈕的正中央,並保持不動！
4...
3...
2...
1...
🎯 測量成功！你電腦上的 HINT 按鈕座標為：
hint_x = 234
hint_y = 681


In [ ]:
import pyautogui as pyg
import time

start_time = time.time()
im = pyg.screenshot()

# 填入你量到的正確 HINT 座標（範例，請替換為你的真實數字）
hint_x, hint_y = 234,681  

# 確保 coordinates 絕不會低於工作列
if hint_y > 980: 
    print("警告:HINT 座標太靠底部，可能點到工作列！請重新量測。") 

# 建立黑名單字典：紀錄每個位置最後一次點擊的時間 {(r, c): timestamp}
click_history = {}

# 設定無效位置的冷卻時間（秒）
COOLDOWN = 3.0  

no_move_count = 0  # 紀錄全盤找不到組合的次數

print("精準機器人啟動！")

while True:
    elapsed = time.time() - start_time
    if elapsed > 60:
        print("60 秒時間到，結束遊戲！")
        break

    moved = False
    current_time = time.time()

    # 掃描 8x8 棋盤[cite: 8]
    for r in range(8):
        for c in range(8):

            # 【關鍵】如果這個位置 3 秒之內才剛剛點過，直接跳過！絕不重複點擊！
            if current_time - click_history.get((r, c), 0) < COOLDOWN:
                continue

            target_r, target_c = None, None

            # 進行 4 個方向的消除計算
            if can_move_left(r, c, im):
                target_r, target_c = r, c - 1
            elif can_move_right(r, c, im):
                target_r, target_c = r, c + 1
            elif can_move_up(r, c, im):
                target_r, target_c = r - 1, c
            elif can_move_down(r, c, im):
                target_r, target_c = r + 1, c

            # 如果計算出可以消除
            if target_r is not None:
                x1, y1 = get_gem_position(r, c)
                x2, y2 = get_gem_position(target_r, target_c)

                # 發動拖曳滑動
                pyg.moveTo(x1, y1)
                pyg.dragTo(x2, y2, duration=0.15, button='left')

                # 記錄這個位置的點擊時間，啟動 3 秒冷卻
                click_history[(r, c)] = current_time
                moved = True
                break

        if moved:
            break

    if moved:
        no_move_count = 0
        # 點擊後給予 0.7 秒等待寶石掉落與動畫結束
        time.sleep(0.7)  
        im = pyg.screenshot()
    else:
        no_move_count += 1
        time.sleep(0.3)
        im = pyg.screenshot()

        # 當因為冷卻黑名單跳過太多的位置，或是畫面上真的卡死時
        # 點一次 HINT 救援，讓遊戲自動清掉一組
        if no_move_count >= 2:
            print("目前無精確解或殘影影響，點擊 HINT 救援以重置盤面...")
            pyg.click(hint_x, hint_y)
            no_move_count = 0
            # 點擊 HINT 後清空黑名單，讓全盤重新計算
            click_history.clear() 
            time.sleep(0.9)
            im = pyg.screenshot()

    print(f"迴圈執行中，已過時間: {elapsed:.1f} 秒")